# 海外债曲线分析

本Notebook用于分析海外债券曲线相关数据。

## 1. 环境配置与依赖导入

In [ ]:
# 导入标准库
import os
import sys
import datetime

# 导入数据处理库
import pandas as pd
import numpy as np

# 导入可视化库
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 导入数据库库
import sqlalchemy
from sqlalchemy import create_engine

# 导入配置
from config import DATABASE_URL, DATA_FILE, PROJECT_NAME

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

print(f"项目: {PROJECT_NAME}")
print(f"当前时间: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 2. 数据加载与预览

In [ ]:
# 加载海外债基本信息数据
if os.path.exists(DATA_FILE):
    df = pd.read_excel(DATA_FILE)
    print(f"数据加载成功，共 {len(df)} 行")
    print(f"\n数据列: {df.columns.tolist()}")
    print(f"\n数据预览:")
    display(df.head(10))
else:
    print(f"数据文件 {DATA_FILE} 不存在，请检查路径")
    df = None

## 3. 数据清洗与处理

In [ ]:
if df is not None:
    # 数据基本信息
    print("数据基本信息:")
    print(df.info())
    
    print("\n数据统计描述:")
    print(df.describe())
    
    # 检查缺失值
    print("\n缺失值统计:")
    missing = df.isnull().sum()
    print(missing[missing > 0])

In [ ]:
# 数据清洗
if df is not None:
    # 处理日期列
    date_columns = df.select_dtypes(include=['object']).columns
    for col in date_columns:
        if '日期' in col or 'date' in col.lower() or 'dt' in col.lower():
            try:
                df[col] = pd.to_datetime(df[col])
                print(f"已转换日期列: {col}")
            except:
                pass
    
    # 处理数值列
    numeric_columns = df.select_dtypes(include=[np.number]).columns
    for col in numeric_columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    print("\n清洗后数据预览:")
    display(df.head())

## 4. 数据分析与可视化

In [ ]:
# 数据分布可视化
if df is not None and len(df) > 0:
    numeric_cols = df.select_dtypes(include=[np.number]).columns[:4]
    
    if len(numeric_cols) > 0:
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        for idx, col in enumerate(numeric_cols):
            if idx >= 4:
                break
            ax = axes[idx // 2, idx % 2]
            df[col].hist(ax=ax, bins=30, edgecolor='black')
            ax.set_title(f'{col} 分布')
            ax.set_xlabel(col)
            ax.set_ylabel('频数')
        
        plt.tight_layout()
        os.makedirs('output', exist_ok=True)
        plt.savefig('output/data_distribution.png', dpi=150, bbox_inches='tight')
        plt.show()

## 5. 债券曲线分析

In [ ]:
# 债券曲线分析
if df is not None:
    # 根据实际数据结构进行分析
    # 示例：按类别统计
    categorical_cols = df.select_dtypes(include=['object']).columns
    
    for col in categorical_cols[:2]:  # 分析前两个分类列
        print(f"\n{col} 分布:")
        print(df[col].value_counts().head(10))

In [ ]:
# 时间序列分析（如果有日期列）
if df is not None:
    date_cols = df.select_dtypes(include=['datetime64']).columns
    
    if len(date_cols) > 0:
        date_col = date_cols[0]
        df_sorted = df.sort_values(date_col)
        
        # 按时间统计
        df_sorted['year'] = df_sorted[date_col].dt.year
        df_sorted['month'] = df_sorted[date_col].dt.month
        
        print("\n按年度统计:")
        print(df_sorted['year'].value_counts().sort_index())

## 6. 数据库操作

In [ ]:
# 数据库连接
def get_db_connection():
    """获取数据库连接"""
    try:
        engine = create_engine(DATABASE_URL, poolclass=sqlalchemy.pool.NullPool)
        conn = engine.connect()
        print("数据库连接成功")
        return engine, conn
    except Exception as e:
        print(f"数据库连接失败: {e}")
        return None, None

engine, conn = get_db_connection()

In [ ]:
# 将数据写入数据库
if df is not None and conn is not None:
    table_name = 'overseas_bond_curve'  # 根据实际需求修改表名
    try:
        df.to_sql(table_name, con=conn, if_exists='replace', index=False)
        print(f"数据已成功写入表: {table_name}")
        print(f"写入记录数: {len(df)}")
    except Exception as e:
        print(f"写入数据库失败: {e}")

## 7. 结果输出与总结

In [ ]:
# 生成输出报告
print("\n" + "="*60)
print(f"项目: {PROJECT_NAME} - 分析报告")
print("="*60)

if df is not None:
    print(f"\n数据处理完成:")
    print(f"  - 原始数据行数: {len(df)}")
    print(f"  - 数据列数: {len(df.columns)}")
    print(f"  - 数值列数: {len(df.select_dtypes(include=[np.number]).columns)}")
    
    # 保存处理后的数据
    output_file = 'output/processed_data.xlsx'
    os.makedirs('output', exist_ok=True)
    df.to_excel(output_file, index=False)
    print(f"\n处理后的数据已保存至: {output_file}")
else:
    print("\n无数据可处理")

print("\n" + "="*60)
print("分析完成！")
print("="*60)

In [ ]:
# 关闭数据库连接
if conn is not None:
    conn.close()
    print("数据库连接已关闭")